# Búsqueda

Muchos problemas tienen la misma forma: llegar de un **estado inicial** a un **estado objetivo** mediante una secuencia de acciones. Una ruta de un pueblo a otro, los movimientos que resuelven un puzle, los pasos de un plan. Los estados y las acciones forman un **grafo**: cada estado es un nodo, y cada acción una arista hacia el estado al que lleva.

Un algoritmo de búsqueda explora ese grafo desde el inicio hasta que llega al objetivo. Los algoritmos de abajo solo se diferencian en **qué nodo exploran a continuación**, y esa elección decide si el camino que devuelven es el más corto, el más barato o simplemente un camino.

## Búsqueda en anchura

La **búsqueda en anchura** (*breadth-first search*, BFS) explora el grafo por niveles: primero todos los nodos a un paso del inicio, después todos los nodos a dos pasos, y así sucesivamente. Guarda los nodos que esperan a ser explorados, la **frontera**, en una cola en la que el primero en entrar es el primero en salir. Como llega a cada nodo con el menor número de pasos posible, el primer camino que encuentra hasta el objetivo es el de **menos pasos**.

Una pequeña red de carreteras, como un diccionario de cada pueblo a sus vecinos:

In [ ]:
road_network = {
    "A": ["C", "B"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F", "G"],
    "F": ["C", "E"],
    "G": ["E"],
}

Cada entrada de la cola es un camino desde el inicio; el último pueblo del camino es el nodo que hay que explorar. `visited` evita que la búsqueda meta un pueblo en la cola dos veces.

In [ ]:
from collections import deque


def breadth_first_search(graph, start, goal):
    frontier = deque([[start]])
    visited = {start}
    explored = []
    while frontier:
        path = frontier.popleft()
        node = path[-1]
        explored.append(node)
        if node == goal:
            return path, explored
        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.add(neighbour)
                frontier.append(path + [neighbour])
    return None, explored


path, explored = breadth_first_search(road_network, "A", "G")
print("Explored:", " ".join(explored))
print("Path:    ", " → ".join(path))

Explored: A C B F D E G
Path:     A → B → E → G


## Búsqueda en profundidad

La **búsqueda en profundidad** (*depth-first search*, DFS) sigue una rama todo lo que puede y solo retrocede al llegar a un callejón sin salida: su frontera es una pila en la que el último en entrar es el primero en salir, aquí la pila de llamadas de una función recursiva. Sin un conjunto `visited` solo necesita memoria para el camino actual, que es su ventaja en espacios de estados enormes; con uno, como aquí, no puede entrar en un bucle infinito en un grafo con ciclos.

Encuentra *un* camino, no necesariamente el más corto: devuelve el primero con el que se topa.

In [ ]:
def depth_first_search(graph, node, goal, path=None, visited=None):
    path = (path or []) + [node]
    visited = visited if visited is not None else set()
    visited.add(node)
    if node == goal:
        return path
    for neighbour in graph[node]:
        if neighbour not in visited:
            found = depth_first_search(graph, neighbour, goal, path, visited)
            if found:
                return found
    return None


print("Path:", " → ".join(depth_first_search(road_network, "A", "G")))

Path: A → C → F → E → G


DFS pasó primero por C, porque C es el primero en la lista de vecinos de A, y llegó al objetivo en cuatro pasos; BFS encontró un camino de tres.

## Búsqueda de coste uniforme

Menos pasos no es lo más barato cuando los pasos tienen costes distintos, como los kilómetros de cada carretera. La **búsqueda de coste uniforme** (*uniform-cost search*, UCS), la versión de búsqueda del algoritmo de Dijkstra, explora siempre el nodo de la frontera con el **menor coste hasta el momento**, que se escribe g(n). Su frontera es una cola de prioridad, y la primera vez que saca el objetivo de la cola, ha encontrado el camino más barato.

Un mapa en el que cada pueblo tiene coordenadas, y cada carretera es al menos tan larga como la línea recta entre sus dos pueblos:

In [ ]:
town_positions = {
    "S": (0, 0), "A": (3, 1), "B": (6, 1), "G": (10, 0), "C": (-3, 0),
    "D": (-3, 4), "E": (0, 5), "F": (5, -3), "H": (8, -3),
}
roads = [
    ("S", "A", 4), ("A", "B", 3), ("B", "G", 5), ("S", "C", 3), ("C", "D", 4),
    ("S", "E", 5), ("E", "D", 4), ("S", "F", 6), ("F", "H", 3), ("H", "G", 4),
]

weighted_network = {town: [] for town in town_positions}
for town_a, town_b, length in roads:
    weighted_network[town_a].append((town_b, length))
    weighted_network[town_b].append((town_a, length))

Ir de S a G cuesta 4 + 3 + 5 = 12 pasando por A y B, y 6 + 3 + 4 = 13 pasando por F y H. C, D y E están detrás del inicio, lejos del objetivo.

In [ ]:
import heapq


def uniform_cost_search(graph, start, goal):
    frontier = [(0, start, [start])]
    expanded = []
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in expanded:
            continue
        expanded.append(node)
        if node == goal:
            return path, cost, expanded
        for neighbour, length in graph[node]:
            if neighbour not in expanded:
                heapq.heappush(frontier, (cost + length, neighbour, path + [neighbour]))
    return None, float("inf"), expanded


path, cost, expanded = uniform_cost_search(weighted_network, "S", "G")
print("Path:", " → ".join(path), "| cost", cost)
print("Expanded", len(expanded), "towns:", " ".join(expanded))

Path: S → A → B → G | cost 12
Expanded 9 towns: S C A E F B D H G


UCS encontró el camino más barato, pero también expandió C, D y E: hace crecer un círculo de coste alrededor del inicio, esté donde esté el objetivo.

## Búsqueda A*

**A\*** añade una estimación del coste que queda, la **heurística** h(n), y explora el nodo con el menor f(n) = g(n) + h(n): el coste hasta el momento más el coste estimado hasta el objetivo. Aquí la heurística es la distancia en línea recta hasta G, que las posiciones de las carreteras dan gratis.

A\* devuelve el camino más barato si la heurística nunca **sobrestima** el coste real que queda (si es **admisible**). Esta versión nunca vuelve a expandir un pueblo, lo que necesita una condición algo más fuerte, la **consistencia**: a lo largo de cada carretera, la estimación no puede bajar más que la longitud de la carretera. Una distancia en línea recta tiene las dos propiedades, porque ninguna carretera es más corta que la línea recta.

In [ ]:
import math


def straight_line_to_goal(town):
    return math.dist(town_positions[town], town_positions["G"])


def a_star(graph, start, goal, heuristic):
    frontier = [(heuristic(start), 0, start, [start])]
    expanded = []
    while frontier:
        _, cost, node, path = heapq.heappop(frontier)
        if node in expanded:
            continue
        expanded.append(node)
        if node == goal:
            return path, cost, expanded
        for neighbour, length in graph[node]:
            if neighbour not in expanded:
                new_cost = cost + length
                heapq.heappush(frontier, (new_cost + heuristic(neighbour), new_cost, neighbour, path + [neighbour]))
    return None, float("inf"), expanded


path, cost, expanded = a_star(weighted_network, "S", "G", straight_line_to_goal)
print("Path:", " → ".join(path), "| cost", cost)
print("Expanded", len(expanded), "towns:", " ".join(expanded))

Path: S → A → B → G | cost 12
Expanded 5 towns: S A B F G


El mismo camino y el mismo coste, con menos pueblos expandidos: la heurística mantuvo a A\* lejos de C, D y E. Con una heurística que siempre vale 0, A\* *es* la búsqueda de coste uniforme:

In [ ]:
path, cost, expanded = a_star(weighted_network, "S", "G", lambda town: 0)
print("Path:", " → ".join(path), "| cost", cost, "| expanded", len(expanded))

Path: S → A → B → G | cost 12 | expanded 9


Una estimación que **sobrestima** rompe la garantía. Duplicar la distancia en línea recta, un truco llamado *A\* ponderado*, hace la búsqueda más voraz: expande todavía menos pueblos, y devuelve un camino que no es el más barato.

In [ ]:
path, cost, expanded = a_star(weighted_network, "S", "G", lambda town: 2 * straight_line_to_goal(town))
print("Path:", " → ".join(path), "| cost", cost, "| expanded", len(expanded))

Path: S → F → H → G | cost 13 | expanded 4


Los planificadores de rutas y los motores de videojuegos hacen exactamente este compromiso: un camino algo peor, encontrado mucho antes.

## Los cuatro algoritmos lado a lado

En la notación habitual, *b* es el factor de ramificación (cuántos sucesores tiene un nodo), *d* la profundidad de la solución menos profunda y *m* la profundidad máxima del grafo.

| Algoritmo | Frontera | Camino que devuelve | Memoria |
| --- | --- | --- | --- |
| En anchura | Cola (FIFO) | El de menos pasos | O(b^d): el nivel entero |
| En profundidad | Pila (LIFO) | Cualquier camino; en grafos finitos encuentra uno si existe | O(b·m) sin conjunto de visitados |
| Coste uniforme | Cola de prioridad por g(n) | El más barato | Hasta O(b^d), y más con pasos baratos |
| A\* | Cola de prioridad por g(n) + h(n) | El más barato, si h es admisible (y consistente, sin reexpansión) | Como UCS en el peor caso; mucho menos con una buena heurística |

El tiempo que tarda cada uno crece como el número de nodos que explora, y por eso importa la heurística: una buena reduce la exploración a poco más que el propio camino.

## Buscar contra un oponente

En un juego, el otro jugador también mueve, así que un plan tiene que funcionar **haga lo que haga el oponente**. **Minimax** busca en el árbol del juego suponiendo que los dos jugadores juegan lo mejor posible: en sus propios turnos elige la jugada con el valor más alto, y en los del oponente la jugada con el más bajo. El tres en raya es lo bastante pequeño (5.478 posiciones posibles) para buscar hasta el final, así que un jugador minimax nunca pierde.

[Python tic-tac-toe](https://github.com/avidaldo/python-tic-tac-toe) construye uno: su nivel 2 implementa minimax, y [MINIMAX_ALGORITHM.md](https://github.com/avidaldo/python-tic-tac-toe/blob/master/02_minimax/MINIMAX_ALGORITHM.md) explica el algoritmo paso a paso.

Los juegos más grandes necesitan más:

- La **poda alfa-beta** se salta las ramas que no pueden cambiar la decisión, y da la misma jugada que minimax con muchísima menos búsqueda.
- La **búsqueda con profundidad limitada** se detiene a una profundidad fija y puntúa las posiciones ahí con una **función de evaluación**, una heurística, como en A\*. Los motores de ajedrez hicieron esto durante décadas con evaluaciones escritas a mano.
- **AlphaGo** (2016) y **AlphaZero** (2017) mantuvieron la búsqueda, como búsqueda en árbol de Monte Carlo, y sustituyeron la evaluación escrita a mano por redes neuronales que la aprendieron de las partidas: búsqueda clásica y aprendizaje automático en un mismo sistema.